# Optimiseurs, taux d’apprentissage et régularisation en pratique

Piloter le réglage des paramètres plutôt qu’augmenter toutes les ressources.

Nous exécutons ce notebook avec un noyau Python neuf, sur CPU.

Version vérifiée : Python 3.11.9, PyTorch 2.10.0 et TorchAudio 2.10.0.

Les cellules n’installent aucun paquet et ne téléchargent aucune donnée.

Les nombres du cours viennent de cette version ; une autre version peut modifier certains arrondis ou messages.

Dans Colab, choisissons un environnement sans accélérateur, puis Exécution → Tout exécuter.

Les trous des exercices restent dans du Markdown : ils ne cassent donc pas l’exécution complète.

La branche CUDA est montrée pour la lecture. L’exécution vérifiée est CPU ; aucune accélération GPU n’est revendiquée.

[Relire le chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#optimiseurs-regularisation-pratique).

Le programme suivant est copié à l’identique depuis le cours.

In [1]:
import os
import sys
import tempfile

# À lancer dans un noyau neuf : ce tutoriel reste sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("CUDA visible :", torch.cuda.is_available())
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-pytorch-")
os.chdir(_quiz_temporary_directory.name)
assert not torch.cuda.is_available(), 'Choisir un noyau CPU neuf pour ce notebook.'

Python : 3.11.9
PyTorch : 2.10.0+cpu
CUDA visible : False


## Nous lisons le programme

**1. Comparer une seule mise à jour**

Les trois optimiseurs partent du même poids et de la même perte. Les valeurs différentes montrent leurs règles, pas une hiérarchie de qualité.

**2. Faire évoluer le taux**

StepLR divise le taux par deux après chaque période de trois époques. Nous contrôlons aussi la validation et conservons son meilleur état.

**3. Distinguer la mesure et la branche optionnelle**

Cette exécution utilise le CPU. Le code active AMP seulement pour CUDA ; aucune accélération ni consommation GPU n’est mesurée ici.

In [2]:
import copy
import torch
from torch import nn

torch.manual_seed(17)
torch.set_num_threads(1)
for classe in [torch.optim.SGD, torch.optim.Adam, torch.optim.AdamW]:
    w = nn.Parameter(torch.tensor(1.))
    opt = classe([w], lr=0.1, weight_decay=0.1)
    (w ** 2).backward()
    opt.step()
    print(classe.__name__, f"{w.item():.6f}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modele = nn.Linear(1, 1).to(device)
x = torch.tensor([[-1.], [0.], [1.]], device=device)
y = 2 * x
vx = torch.tensor([[-0.5], [0.5]], device=device)
vy = 2 * vx
opt = torch.optim.AdamW(modele.parameters(), lr=0.1, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=3, gamma=0.5)
amp_actif = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=amp_actif)
meilleure, attente, meilleur_etat = float("inf"), 0, None
for epoch in range(1, 9):
    modele.train()
    opt.zero_grad(set_to_none=True)
    with torch.amp.autocast(device_type=device.type, enabled=amp_actif):
        perte = nn.functional.mse_loss(modele(x), y)
    scaler.scale(perte).backward()
    scaler.step(opt)
    scaler.update()
    modele.eval()
    with torch.no_grad():
        val = nn.functional.mse_loss(modele(vx), vy).item()
    if val < meilleure - 1e-5:
        meilleure, attente = val, 0
        meilleur_etat = copy.deepcopy(modele.state_dict())
    else:
        attente += 1
    print(f"epoch {epoch}: val={val:.6f}, lr={opt.param_groups[0]['lr']:.5f}")
    scheduler.step()
    if attente >= 2:
        break
modele.load_state_dict(meilleur_etat)
print("Calcul :", device.type, "| AMP active :", amp_actif)
print("Octets des paramètres :", sum(p.numel() * p.element_size() for p in modele.parameters()))
if device.type == "cuda":
    print("Octets CUDA alloués :", torch.cuda.memory_allocated(device))

SGD 0.790000
Adam 0.900000
AdamW 0.890000
epoch 1: val=1.032523, lr=0.10000
epoch 2: val=0.936570, lr=0.10000
epoch 3: val=0.840762, lr=0.10000
epoch 4: val=0.794807, lr=0.05000
epoch 5: val=0.750981, lr=0.05000
epoch 6: val=0.709355, lr=0.05000
epoch 7: val=0.689177, lr=0.02500
epoch 8: val=0.669253, lr=0.02500
Calcul : cpu | AMP active : False
Octets des paramètres : 8


## Interpréter avant de modifier

Exécution CPU vérifiée ; les sorties sont conservées dans scripts/verify/pytorch-additions-results.json.

Un optimiseur définit une règle, pas une performance universelle.

Scheduler et arrêt anticipé agissent sur des décisions différentes.

Une branche CUDA présente dans le code ne constitue pas une mesure GPU.

## Exercice 1 — Contrôler une mise à jour SGD

Nous minimisons w² à partir de w=1. Complétons une étape SGD sans régularisation, avec lr=0,1.

Copions ce squelette dans une nouvelle cellule, puis remplaçons les `...`.

```python
w = torch.nn.Parameter(torch.tensor(1.))
opt = torch.optim.SGD([w], lr=...)
opt....(set_to_none=True)
w.square().backward()
opt....()
print(w.item())
```

**Piste :** Le gradient vaut deux. La règle soustrait lr × gradient.

## Exercice 2 — Séparer scheduler et arrêt anticipé

Une validation s’améliore puis stagne. Complétons le compteur d’attente pour une patience de deux observations.

Copions ce squelette dans une nouvelle cellule, puis remplaçons les `...`.

```python
meilleure = float('inf')
attente = 0
for epoch, val in enumerate([0.5, 0.4, 0.41, 0.42], start=1):
    if val < meilleure:
        meilleure, attente = val, ...
    else:
        attente += ...
    if attente >= ...:
        print('arret', epoch)
        break
```

**Piste :** Ce compteur décide quand arrêter. Un scheduler déciderait comment modifier le taux d’apprentissage.

In [3]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()
print('Programme terminé ; fichiers temporaires supprimés.')

Programme terminé ; fichiers temporaires supprimés.


## Solutions — ouvrons-les après notre recherche

Les solutions utilisent leurs propres variables. Nous pouvons les copier dans de nouvelles cellules.

<details>
<summary>Afficher la solution 1 — Contrôler une mise à jour SGD</summary>

<pre><code>import torch
w_exo = torch.nn.Parameter(torch.tensor(1.))
opt_exo = torch.optim.SGD([w_exo], lr=0.1)
opt_exo.zero_grad(set_to_none=True)
w_exo.square().backward()
opt_exo.step()
assert torch.allclose(w_exo, torch.tensor(0.8))
print(&#x27;poids après SGD :&#x27;, w_exo.item())</code></pre>

</details>

<details>
<summary>Afficher la solution 2 — Séparer scheduler et arrêt anticipé</summary>

<pre><code>meilleure_exo, attente_exo = float(&#x27;inf&#x27;), 0
arret_exo = None
for epoch_exo, val_exo in enumerate([0.5, 0.4, 0.41, 0.42], start=1):
    if val_exo &lt; meilleure_exo:
        meilleure_exo, attente_exo = val_exo, 0
    else:
        attente_exo += 1
    if attente_exo &gt;= 2:
        arret_exo = epoch_exo
        break
assert arret_exo == 4 and meilleure_exo == 0.4
print(&#x27;arrêt à l’époque :&#x27;, arret_exo)</code></pre>

</details>

